https://mlipper.github.io/geoclient/#borough

Geoclient's `borough` parameter accepts, per the docs above: `Manhattan`/`MN`/`1`, `Bronx`/`BX`/`The Bronx`/`2`, `Brooklyn`/`BK`/`BKLYN`/`3`, `Queens`/`QN`/`4`, `Staten Island`/`SI`/`STATENISLAND`/`STATENIS`/`5`.

`tickets_all.violation_county` mixes two different abbreviation schemes across fiscal years: the colloquial `MN`/`BK`/`QN`/`ST`/`BX`, and the older DOF county codes `NY` (New York County), `K` (Kings), `Q` (Queens), `R` (Richmond) - plus `ST` for Staten Island, which isn't one of Geoclient's accepted aliases. Build a map from every observed value to one accepted Geoclient borough name.

In [1]:
import sys
sys.path.append("..")

from data.base import get_connection

county_counts = get_connection().sql("""
    SELECT violation_county, COUNT(*) AS tickets
    FROM tickets_all
    GROUP BY violation_county
    ORDER BY tickets DESC
""").df()

county_counts

,violation_county,tickets
0,QN,16888926
1,BK,15266912
2,BX,6829793
3,ST,3690076
4,MN,3165874
5,NaN,39270
6,NY,23
7,K,9
8,R,1


In [3]:
# map every violation_county value seen in tickets_all to one Geoclient-accepted
# borough name. NaN (missing county) has no safe mapping and is left out, so
# those tickets can't be geocoded via /v2/intersection without another source
# of borough info.
VIOLATION_COUNTY_TO_BOROUGH = {
    "MN": "Manhattan",
    "NY": "Manhattan",      # DOF county code for New York County
    "BX": "Bronx",
    "BK": "Brooklyn",
    "K": "Brooklyn",        # DOF county code for Kings County
    "QN": "Queens",
    "Q": "Queens",          # DOF county code for Queens County
    "ST": "Staten Island",
    "R": "Staten Island",   # DOF county code for Richmond County
}

unmapped = set(county_counts["violation_county"].dropna()) - set(VIOLATION_COUNTY_TO_BOROUGH)
assert not unmapped, f"unmapped violation_county values: {unmapped}"

missing_county = county_counts.loc[county_counts["violation_county"].isna(), "tickets"].sum()
print(f"{missing_county} tickets have no violation_county and can't be mapped to a borough")

39270 tickets have no violation_county and can't be mapped to a borough
